In [9]:
import yaml
import pandas as pd
from pathlib import Path
import glob

# Base directory for optimization results
opt_results_dir = Path("opt_results") / "run_combined_sim"

# Define methods and their directories
methods = ["graph_ga", "gp_bo", "smiles_ga", "reinvent"]
seeds = [0]
oracle_name = "oracle"

def convert_yaml_to_csv(yaml_path, csv_path):
    """
    Convert YAML optimization results to CSV format.
    
    Args:
        yaml_path: Path to input YAML file
        csv_path: Path to output CSV file
    """
    # Load YAML file
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    
    # Convert to list of dictionaries
    records = []
    for smiles, values in data.items():
        records.append({
            'smiles': smiles,
            'rnamigos2_score': values[0],  # First value is the score
            'iteration': values[1]          # Second value is the iteration
        })
    
    # Create DataFrame and save to CSV
    df = pd.DataFrame(records)
    df.to_csv(csv_path, index=False)
    print(f"Converted: {yaml_path.name} -> {csv_path.name} ({len(df)} molecules)")
    
    return df

# Process all methods and seeds
print("=" * 80)
print("Converting YAML files to CSV format")
print("=" * 80)

summary = []

for method in methods:
    method_dir = opt_results_dir / method
    
    if not method_dir.exists():
        print(f"Skipping {method} - directory not found")
        continue
    
    print(f"\n📁 Processing method: {method}")
    print("-" * 80)
    
    for seed in seeds:
        yaml_file = method_dir / f"results_{method}_{oracle_name}_{seed}.yaml"
        csv_file = method_dir / f"results_{method}_{oracle_name}_{seed}.csv"
        
        if yaml_file.exists():
            df = convert_yaml_to_csv(yaml_file, csv_file)
            
            # Collect summary statistics
            summary.append({
                'method': method,
                'seed': seed,
                'num_molecules': len(df),
                'max_score': df['rnamigos2_score'].max(),
                'mean_score': df['rnamigos2_score'].mean(),
                'min_score': df['rnamigos2_score'].min(),
                'max_iteration': df['iteration'].max()
            })
        else:
            print(f"⚠️  File not found: {yaml_file.name}")

# Create summary DataFrame
print("\n" + "=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)
summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))


Converting YAML files to CSV format

📁 Processing method: graph_ga
--------------------------------------------------------------------------------


Converted: results_graph_ga_oracle_0.yaml -> results_graph_ga_oracle_0.csv (8449 molecules)

📁 Processing method: gp_bo
--------------------------------------------------------------------------------
Converted: results_gp_bo_oracle_0.yaml -> results_gp_bo_oracle_0.csv (10001 molecules)

📁 Processing method: smiles_ga
--------------------------------------------------------------------------------
Converted: results_smiles_ga_oracle_0.yaml -> results_smiles_ga_oracle_0.csv (10001 molecules)

📁 Processing method: reinvent
--------------------------------------------------------------------------------
Converted: results_reinvent_oracle_0.yaml -> results_reinvent_oracle_0.csv (9001 molecules)

SUMMARY STATISTICS
   method  seed  num_molecules  max_score  mean_score  min_score  max_iteration
 graph_ga     0           8449   0.393239   -0.037986  -0.547388           8449
    gp_bo     0          10001   1.341004    0.286602  -0.436940          10001
smiles_ga     0          10001   0.60218